# Group 7 — Science Communication Channels and Public Trust: Analysis Notebook

Rendered companion to the canonical analysis script; all outputs were produced by executing this notebook top-to-bottom against the full TISP dataset (N = 71,922 respondents, 68 countries).

Canonical script version: [`code/analysis.py`](../code/analysis.py)

In [ ]:
import os, sys

# Resolve repo root so relative paths (data/, figures/, code/) work
# regardless of whether Jupyter was launched from the repo root or from notebooks/
_cwd = os.getcwd()
_repo_root = _cwd if os.path.isdir(os.path.join(_cwd, 'code')) else os.path.dirname(_cwd)
os.chdir(_repo_root)
sys.path.insert(0, os.path.join(_repo_root, 'code'))

from analysis import load, multilevel_model, sem_mediation, choropleth, anova_by_channel

In [ ]:
df = load('data/raw/ds_main.csv')
print()
print('Dataset shape :', df.shape)
print('Countries     :', df['country'].nunique())

## 1. Multi-level Model + ICC

Mixed-effects model with random intercepts by country. ICC quantifies the proportion of trust variance attributable to country-level factors.

In [ ]:
result, icc = multilevel_model(df)

## 2. SEM Mediation

Structural equation model testing whether science knowledge mediates the relationship between communication frequency and trust in scientists (comm_freq → knowledge → trust).

In [ ]:
sem_out = sem_mediation(df)

## 3. Geographic Visualisation — Choropleth

Mean trust in scientists plotted by country. Higher-income regions (Northern/Western Europe, North America, East Asia) cluster at higher trust scores.

In [ ]:
import plotly.express as px
from IPython.display import Image, display

country_means = choropleth(df)
# Inline PNG for notebook rendering; analysis (country-level means) is performed by choropleth() above
_fig = px.choropleth(
    country_means, locations='country', color='mean_trust',
    color_continuous_scale='RdYlBu', locationmode='ISO-3',
    title='Mean Trust in Scientists by Country (TISP, N=71,922, 68 Countries)',
    labels={'mean_trust': 'Mean Trust (1-5)'},
)
_fig.write_image('figures/trust_choropleth_nb.png', width=900, height=500)
display(Image('figures/trust_choropleth_nb.png'))

## 4. ANOVA — Trust by Primary Communication Channel

One-way ANOVA testing whether mean trust in scientists differs significantly across respondents’ primary science information channel.

In [ ]:
from IPython.display import Image, display

f_stat, p_val = anova_by_channel(df)
display(Image('figures/anova_channel_boxplot.png'))

## Key Findings

| Analysis | Headline result |
|---|---|
| **Multi-level model** | **ICC = 0.105** — 10.5 % of variance in trust in scientists lies between countries, justifying the multi-level approach |
| **SEM mediation** | **Indirect effect = 0.045** — science knowledge partially mediates the positive association between communication frequency and trust |
| **ANOVA** | **F = 27.963, p < 0.001** — trust differs significantly across primary communication channels |
| **Choropleth** | Higher-income regions (Northern/Western Europe, North America, East Asia) show the highest mean trust; sub-Saharan Africa and parts of South Asia show the lowest |

All analyses use the full TISP analytic sample (observations with non-missing trust, comm_freq, and knowledge); see [`code/analysis.py`](../code/analysis.py) for composite-measure derivations.